#### PUNTO 1.A


##### La dinamica del motor de continua, se rige por las siguientes ecuaciones diferenciales:
$$
\begin{cases} 
L\tfrac{di(t)}{dt}+Ri(t)+K_e\omega(t)=V(t) & (1)\\
J\tfrac{d\omega(t)}{dt}+B\omega(t)=\tau_m(t)=K_ti(t) & (2)\\
\omega(t)=\tfrac{d\theta(t)}{dt} & (3)
\end{cases}
$$

##### Aplicamos Laplace a las tres EDOs:
$$
\begin{cases} 
LsI(s)+RI(s)+K_e\Omega(s)=V(s) & (4)\\
Js\Omega(s)+B\Omega(s)=K_tI(s) & (5)\\
\Omega(s)=s\varTheta(s) & (6)
\end{cases}
$$

##### Reemplazando (3) en (2) se obtiene $I(s)=\tfrac{\varTheta(s)(Js^2+Bs)}{K_t}$. Esto se usa en (1) y se obtiene $\tfrac{\varTheta(s)}{V(s)}=\tfrac{K_t}{s(LJs^2+(JR+BL)s+(BR+K_eK_t))}$

##### Ecuaciones en variables de estado:
$$
\begin{cases} 
x_1=\theta(t) \\
x_2=\omega(t) \\
x_3=i(t)
\end{cases}
$$

$$
\begin{cases} 
\dot x_1=x_2 \\
\dot x_2=\tfrac{k_tx_3-Bx_2}{J} \\
\dot x_3=\tfrac{V(t)-K_ex_2-Rx_3}{L}
\end{cases}
$$


##### El sistema se expresa en forma matricial como $\dot{x} = Ax + Bu$ y $y = Cx$:

$$
\begin{bmatrix}
\dot x_1 \\
\dot x_2 \\
\dot x_3
\end{bmatrix}
=
\begin{bmatrix}
0 & 1 & 0 \\
0 & -\tfrac{B}{J} & \tfrac{K_t}{J} \\
0 & -\tfrac{K_e}{L} & -\tfrac{R}{L}
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3
\end{bmatrix}
+
\begin{bmatrix}
0 \\
0 \\
\tfrac{1}{L}
\end{bmatrix} V(t)\\
y =
\begin{bmatrix}
1 & 0 & 0
\end{bmatrix}
\begin{bmatrix}
x_1 \\
x_2 \\
x_3
\end{bmatrix}
+
\begin{bmatrix}
0
\end{bmatrix} V(t)
$$

In [ ]:
import control as ctrl
import matplotlib.pyplot as plt
import numpy as np

##### Para punto 1

In [ ]:
L = 0.15
R = 0.7
K_e = 0.1
J = 0.02
B = 0.25
K_t = 0.25

# Funcion de transferencia
G1 = ctrl.tf([K_t], [L*J, J*R + B*L, B*R + K_e*K_t, 0])

# Forma matricial
A = np.array ([[0, 1, 0], [0, -B/J, K_t/J], [0, -K_e/L, -R/L]])
B = np.array ([[0], [0], [1/L]])
C = np.array ([[1, 0, 0]])
D = np.array ([[0]])

#### PUNTO 2.B

In [ ]:
def create_closed_loop_system(K, J, B_val):
    num = [K]
    den = [J, B_val, 0]
    G = ctrl.tf(num, den)
    return ctrl.feedback(G, 1)

K = 1
J = 1

In [ ]:
# Tiempos de simulacion
t1 = np.linspace(0, 15, 1000)
t2 = np.linspace(15, 90, 3000)

G_B05_closed = create_closed_loop_system(K, J, 0.5)
G_B01_closed = create_closed_loop_system(K, J, -0.1)

# Sistema con B=0.5
sys1 = ctrl.ss(G_B05_closed)
response1 = ctrl.forced_response(sys1, T=t1, U=1)
t_out1 = response1.time
y_out1 = response1.outputs
x_out1 = response1.states
x_final_stage1 = x_out1[:, -1]

# Sistema con B=-0.1
sys2 = ctrl.ss(G_B01_closed)
response2 = ctrl.forced_response(sys2, T=t2, U=1, X0=x_final_stage1)
t_out2 = response2.time
y_out2 = response2.outputs
x_out2 = response2.states

t_total = np.concatenate([t_out1, t_out2])
y_total = np.concatenate([y_out1, y_out2])

plt.figure(figsize=(10, 5))
plt.plot(t_total, y_total, label='Respuesta del sistema', color='red')
plt.axvline(x=15, color='black', linestyle='--', label='Instante de la falla')
plt.xlabel('Tiempo [s]')
plt.ylabel('Salida θ(t)')
plt.title('Simulacion de falla fisica en el motor')
plt.grid(True)
plt.legend()
plt.show()

#### PUNTO 2C

In [ ]:
G_B05 = ctrl.tf([K], [J, 0.5, 0])
G_B01 = ctrl.tf([K], [J, -0.1, 0])

ctrl.root_locus(G_B05)
plt.xlabel('Parte real')
plt.ylabel('Parte imaginaria')
plt.xlim(-1.2, 1.2) 
plt.ylim(-1.2, 1.2)
plt.grid(True)
plt.show()

ctrl.root_locus(G_B01)
plt.xlabel('Parte real')
plt.ylabel('Parte imaginaria')
plt.xlim(-1.2, 1.2) 
plt.ylim(-1.2, 1.2)
plt.grid(True)
plt.show()   


##### La estabilidad depende del valor de B: la función de transferencia es $\tfrac{K}{s(Js+B)}=\tfrac{1}{s(s+B)}$, los polos del sistema son $s=0$ y $s=-B$.
##### De esta manera:
- Con $B>0$ el polo tiene parte real negativa, por lo que el sistema es estable.
- Con $B<0$ el polo tiene parte real positiva, por lo que el sistema es inestable.
- Con $B=0$ hay doble polo en cero, po rlo que el sistema se vuelve marginalmente estable (oscila).

### <span style="color:red">Falta ROBUSTEZ del sistema frente a variaciones en el parametro B</span>